# Pipeline de hitos 3D (.glb) · Muro de Boxes (Google Colab)

Genera hitos icónicos de los 24 circuitos en una GPU gratuita de Google Colab (T4) y los optimiza automáticamente al contrato PWA del juego:
- **Geometría:** ~3 000–6 000 triángulos (decimado automático).
- **Texturas:** WebP de hasta **512 × 512 px**.
- **Compresión:** **Draco**, objetivo **< 300 KB** por hito.
- **Normalización:** base apoyada en `y = 0`, centrado en `X/Z` y altura normalizada a `1,0 m` (`map3d.ts` lo escala a `spec.height`).

Al terminar, descarga `landmarks.zip` y descomprímelo dentro de `public/assets/landmarks/` del repositorio.

In [ ]:
# 1. Comprobar GPU e instalar dependencias de optimización (Draco + WebP + decimado)
!nvidia-smi -L
!pip -q install trimesh pillow numpy fast-simplification huggingface_hub
!npm install -g --silent @gltf-transform/cli

In [ ]:
# 2. Elegir qué hitos generar (id del circuito, id del hito y prompt en inglés)
# El catálogo completo de los 24 circuitos está en src/ui/landmarks.ts
HITOS = [
    ("suzuka", "noria", "Stylized low-poly white and red Suzuka Circuit Ferris wheel with A-frame support and gondolas, game asset"),
    ("austin", "torre_cota", "Stylized low-poly Circuit of the Americas COTA observation tower with sweeping red steel veil and white top deck, game asset"),
    ("lasvegas", "esfera", "Stylized low-poly Las Vegas MSG Sphere giant glowing LED dome arena in cyan and magenta, game asset"),
    ("singapur", "marina_sands", "Stylized low-poly Marina Bay Sands Singapore three skyscrapers topped by boat SkyPark deck, game asset"),
    ("baku", "flame_towers", "Stylized low-poly Baku Flame Towers trio of curved flame-shaped glass skyscrapers, game asset"),
    ("yasmarina", "hotel_w", "Stylized low-poly W Abu Dhabi Yas Marina hotel curved gridshell glass canopy, game asset"),
    ("spielberg", "toro_acero", "Stylized low-poly giant stainless steel and crimson charging bull sculpture on stone plinth, game asset"),
    ("miami", "hard_rock_stadium", "Stylized low-poly Hard Rock Stadium Miami with white open canopy and four corner white spires, game asset"),
]
TARGET_TRIS = 4500
MAX_TEX_PX = 512

In [ ]:
# 3A. Opción A (Imagen -> 3D con Hunyuan3D-2 en GPU T4) o 3B (subir .glb/.obj desde Blender/Meshy)
# Por defecto crea la carpeta raw_models/ para que puedas generar con Hunyuan3D-2 o subir tus .glb brutos.
import os
os.makedirs("raw_models", exist_ok=True)
os.makedirs("out_landmarks", exist_ok=True)

MODO = "subir"  # Cambia a "hunyuan" para instalar y ejecutar Tencent Hunyuan3D-2 en GPU T4

if MODO == "hunyuan":
    !git clone -q https://github.com/Tencent/Hunyuan3D-2.git
    !pip -q install -r Hunyuan3D-2/requirements.txt diffusers transformers accelerate
    # Genera primero una vista ortogonal limpia (SDXL-Turbo / Flux) y pásala a Hunyuan3D-2
    print("Hunyuan3D-2 listo: coloca o genera una imagen PNG por hito en raw_models/<circuito>_<hito>.png y ejecuta la malla.")
else:
    from google.colab import files
    print("Sube tus archivos brutos nombrados <circuito>__<hito>.glb (p. ej. suzuka__noria.glb):")
    uploaded = files.upload()
    for fn, data in uploaded.items():
        open(os.path.join("raw_models", fn), "wb").write(data)

In [ ]:
# 4. Optimizar todos los archivos de raw_models/ al contrato del juego (<300 KB, Draco + WebP 512px, pivote y=0)
import glob, json, os, shutil, subprocess, tempfile
import numpy as np
import trimesh
import fast_simplification

manifest = {"version": 1, "glb": {}}

for src in sorted(glob.glob("raw_models/*.glb") + glob.glob("raw_models/*.obj")):
    base = os.path.splitext(os.path.basename(src))[0]
    if "__" not in base:
        print(f"Saltando {src}: usa el formato <circuito>__<hito>.glb")
        continue
    cid, hid = base.split("__", 1)
    loaded = trimesh.load(src, force="scene")
    scene = loaded if isinstance(loaded, trimesh.Scene) else trimesh.Scene(loaded)
    n_before = sum(len(g.faces) for g in scene.geometry.values() if isinstance(g, trimesh.Trimesh))
    if n_before > TARGET_TRIS:
        for k, g in list(scene.geometry.items()):
            if isinstance(g, trimesh.Trimesh) and len(g.faces) > 64:
                share = max(64, int(round(TARGET_TRIS * len(g.faces) / n_before)))
                ratio = max(0.05, min(0.95, 1.0 - share / float(len(g.faces))))
                v, f = fast_simplification.simplify(np.asarray(g.vertices, np.float32), np.asarray(g.faces, np.int32), target_reduction=ratio)
                scene.geometry[k] = trimesh.Trimesh(vertices=v, faces=f, process=True)
    # Normalizar pivote (base en y = 0, centrado en X/Z, altura = 1 m)
    lo, hi = scene.bounds
    T = np.eye(4); T[:3, 3] = [-0.5*(lo[0]+hi[0]), -lo[1], -0.5*(lo[2]+hi[2])]
    s = 1.0 / max(1e-4, float(hi[1] - lo[1]))
    scene.apply_transform(np.diag([s, s, s, 1.0]) @ T)
    out_dir = os.path.join("out_landmarks", cid)
    os.makedirs(out_dir, exist_ok=True)
    out_glb = os.path.join(out_dir, f"{hid}.glb")
    with tempfile.TemporaryDirectory() as td:
        raw_glb = os.path.join(td, "raw.glb")
        webp_glb = os.path.join(td, "webp.glb")
        scene.export(raw_glb, file_type="glb")
        subprocess.run(["gltf-transform", "webp", raw_glb, webp_glb, "--resize", f"{MAX_TEX_PX},{MAX_TEX_PX}", "--quality", "80"], check=False)
        cur = webp_glb if os.path.exists(webp_glb) else raw_glb
        subprocess.run(["gltf-transform", "draco", cur, out_glb, "--method", "edgebreaker"], check=True)
    n_after = sum(len(g.faces) for g in scene.geometry.values() if isinstance(g, trimesh.Trimesh))
    kb = os.path.getsize(out_glb) / 1024.0
    manifest["glb"][f"{cid}/{hid}"] = True
    print(f"{cid}/{hid}: {n_before} -> {n_after} tris | {kb:.1f} KB {'OK' if kb <= 300 else '! >300 KB'}")

open("out_landmarks/manifest.json", "w", encoding="utf8").write(json.dumps(manifest, indent=2, ensure_ascii=False) + "\n")
shutil.make_archive("landmarks", "zip", "out_landmarks")
print("Listo: landmarks.zip preparado para descomprimir en public/assets/landmarks/")

In [ ]:
# 5. Descargar el paquete listo para el repositorio
from google.colab import files
files.download("landmarks.zip")